# Fase 1 — Entendimento dos Dados

**Dataset:** IEEE-CIS Fraud Detection (Kaggle)  
**Objetivo desta fase:** Entender a estrutura das duas tabelas (`transaction` e `identity`), o significado dos grupos de variáveis, os valores ausentes e o merge entre as tabelas.

---

## Contexto do Problema

Os dados foram fornecidos pela **Vesta Corporation**, empresa de prevenção de fraudes. Cada linha representa uma **transação financeira** (geralmente um pagamento por e-commerce). O objetivo é prever se uma transação é fraudulenta (`isFraud = 1`) ou legítima (`isFraud = 0`).

### As duas tabelas

| Tabela | Linhas | Colunas | Conteúdo |
|--------|--------|---------|----------|
| `train_transaction` | ~590k | 394 | Dados financeiros da transação |
| `train_identity` | ~141k | 41 | Dados de identidade do dispositivo/usuário |

Nem toda transação tem dados de identidade — apenas ~24% das linhas de `transaction` têm uma linha correspondente em `identity`. Faremos um **LEFT JOIN** para preservar todas as transações.

### Por que LEFT JOIN e não INNER JOIN?

- **INNER JOIN** manteria apenas as ~141k transações que têm identidade → perderíamos ~450k transações, inclusive fraudes
- **LEFT JOIN** mantém todas as ~590k transações, preenchendo com `NaN` onde não há dados de identidade
- No pré-processamento (Fase 3), trataremos esses `NaN` adequadamente

## 1. Imports e Configuração

In [ ]:
import os
import sys
import warnings

import pandas as pd
import numpy as np

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.4f}'.format)

# Adiciona o diretório raiz ao path para importar src/utils.py
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.insert(0, PROJECT_ROOT)

from src.utils import load_data, merge_tables, missing_summary

# Caminho para os dados brutos (ajuste se necessário)
DATA_DIR = '/home/fernando-daniel-marcelino/data/ieee-cis-fraud-detection'

print(f'Pandas version : {pd.__version__}')
print(f'NumPy version  : {np.__version__}')
print(f'Data dir       : {DATA_DIR}')

## 2. Carregamento dos Dados

Os arquivos CSV somam ~1.3 GB no disco. O carregamento pode levar 30–60 segundos dependendo da máquina — é normal.

In [ ]:
df_trans, df_id = load_data(DATA_DIR)

## 3. Inspeção Inicial — Tabela de Transações

A tabela de transações é a principal. Vamos inspecionar sua estrutura antes de qualquer outra coisa.

In [ ]:
print('=== TABELA DE TRANSAÇÕES ===')
print(f'Linhas: {df_trans.shape[0]:,}')
print(f'Colunas: {df_trans.shape[1]}')
print(f'Memória: {df_trans.memory_usage(deep=True).sum() / 1e6:.1f} MB')

In [ ]:
# Primeiras linhas — só para ter um senso visual dos dados
df_trans.head(3)

### Grupos de variáveis da tabela de transações

O IEEE-CIS organizou as ~394 colunas em grupos com significados diferentes. Entender esses grupos é fundamental antes de qualquer modelagem:

| Grupo | Colunas | Descrição |
|-------|---------|----------|
| **ID** | `TransactionID` | Chave única da transação |
| **Target** | `isFraud` | 0 = legítimo, 1 = fraude |
| **Tempo** | `TransactionDT` | Segundos desde um ponto de referência (não é timestamp real) |
| **Valor** | `TransactionAmt` | Valor em USD da transação |
| **Produto** | `ProductCD` | Categoria do produto (W, H, C, S, R) |
| **Cartão** | `card1`–`card6` | Informações do cartão (banco emissor, tipo, país etc.) |
| **Endereço** | `addr1`, `addr2` | Endereço de cobrança (código de área, país) |
| **Distância** | `dist1`, `dist2` | Distâncias entre endereços associados à transação |
| **E-mail** | `P_emaildomain`, `R_emaildomain` | Domínio do e-mail do comprador (P) e destinatário (R) |
| **Contagem** | `C1`–`C14` | Contadores de comportamento (ex: quantos cartões o endereço usou) |
| **Delta tempo** | `D1`–`D15` | Dias desde eventos anteriores (ex: última transação com o mesmo cartão) |
| **Match** | `M1`–`M9` | Flags booleanas de correspondência (ex: nome no cartão bate com o cadastro?) |
| **Vesta** | `V1`–`V339` | Features proprietárias da Vesta — significado não divulgado |

> **Nota sobre V1–V339:** A Vesta não divulgou o significado dessas features por questões de propriedade intelectual. Mesmo sem saber o que são, o modelo consegue aprender padrões a partir delas. Durante a interpretabilidade (Fase 5 com SHAP), veremos quais delas mais impactam as previsões.

In [ ]:
# Categorizar as colunas por grupo para facilitar análises futuras
col_groups = {
    'id':        ['TransactionID'],
    'target':    ['isFraud'],
    'time':      ['TransactionDT'],
    'amount':    ['TransactionAmt'],
    'product':   ['ProductCD'],
    'card':      [c for c in df_trans.columns if c.startswith('card')],
    'addr':      [c for c in df_trans.columns if c.startswith('addr')],
    'dist':      [c for c in df_trans.columns if c.startswith('dist')],
    'email':     ['P_emaildomain', 'R_emaildomain'],
    'C_count':   [c for c in df_trans.columns if c.startswith('C') and c[1:].isdigit()],
    'D_delta':   [c for c in df_trans.columns if c.startswith('D') and c[1:].isdigit()],
    'M_match':   [c for c in df_trans.columns if c.startswith('M') and c[1:].isdigit()],
    'V_vesta':   [c for c in df_trans.columns if c.startswith('V') and c[1:].isdigit()],
}

print('Contagem de colunas por grupo:')
for group, cols in col_groups.items():
    print(f'  {group:10s}: {len(cols):3d} coluna(s)')

In [ ]:
# Tipos de dados por grupo — importante para saber o que precisará de encoding
print('Tipos de dados (por grupo):\n')
for group, cols in col_groups.items():
    dtypes = df_trans[cols].dtypes.value_counts()
    print(f'  {group:10s}: {dict(dtypes)}')

## 4. Inspeção Inicial — Tabela de Identidade

A tabela de identidade contém informações sobre o dispositivo e a rede usados na transação.

In [ ]:
print('=== TABELA DE IDENTIDADE ===')
print(f'Linhas: {df_id.shape[0]:,}')
print(f'Colunas: {df_id.shape[1]}')
print(f'Memória: {df_id.memory_usage(deep=True).sum() / 1e6:.1f} MB')

df_id.head(3)

In [ ]:
# Grupos de variáveis da tabela de identidade
id_col_groups = {
    'id_link':   ['TransactionID'],
    'id_num':    [c for c in df_id.columns if c.startswith('id_') and c[3:].isdigit()],
    'device':    ['DeviceType', 'DeviceInfo'],
}

print('Colunas de identidade por grupo:')
for group, cols in id_col_groups.items():
    print(f'  {group:10s}: {len(cols):3d} coluna(s) — {cols[:5]}{"..." if len(cols) > 5 else ""}')

print('\nTipos de dados:')
print(df_id.dtypes.value_counts())

### Variáveis de identidade (id_01 a id_38)

Assim como as variáveis V, a maioria das `id_XX` foi anonimizada. O que se sabe:

- `id_01`–`id_11`: Numéricas — capturam informações de rede e comportamento (ex: score de risco de IP, proxy detection)
- `id_12`–`id_38`: Categóricas — capturam informações do dispositivo, sistema operacional, navegador
- `DeviceType`: `'desktop'` ou `'mobile'` — tipo de dispositivo
- `DeviceInfo`: Modelo específico do dispositivo (ex: `'Windows'`, `'iOS Device'`, `'Samsung SM-...'`)

## 5. Variável Target — isFraud

Antes de qualquer outra análise, precisamos entender o que estamos prevendo.

In [ ]:
fraud_counts = df_trans['isFraud'].value_counts()
fraud_pct    = df_trans['isFraud'].value_counts(normalize=True) * 100

print('Distribuição da variável target (isFraud):')
print(f'  Legítimas (0): {fraud_counts[0]:>7,} transações  ({fraud_pct[0]:.2f}%)')
print(f'  Fraudes   (1): {fraud_counts[1]:>7,} transações  ({fraud_pct[1]:.2f}%)')
print(f'  Total         : {len(df_trans):>7,} transações')
print(f'\nRazão de desbalanceamento: {fraud_counts[0]/fraud_counts[1]:.1f}:1')
print('(Para cada transação fraudulenta, há ~X legítimas)')

> **Interpretação:** O desbalanceamento de ~96.5% vs ~3.5% é um problema clássico em fraude. Se um modelo simplesmente chutasse "legítimo" para tudo, teria 96.5% de acurácia — mas seria inútil. Por isso, na Fase 4 usaremos métricas como **AUC-ROC** e **Precision-Recall**, e técnicas como `class_weight='balanced'` ou SMOTE na Fase 3.

## 6. Análise de Valores Ausentes

Valores ausentes (`NaN`) são muito comuns neste dataset — especialmente nas variáveis V e nas variáveis de identidade. Entender o padrão de missing é crucial para decidir como tratá-los na Fase 3.

In [ ]:
# Resumo de missings da tabela de transações
missing_trans = missing_summary(df_trans)

print(f'Colunas com valores ausentes: {len(missing_trans)} de {df_trans.shape[1]}')
print('\nTop 20 colunas com mais missing:')
missing_trans.head(20)

In [ ]:
# Distribuição do % de missing por faixas
missing_pct_numeric = df_trans.isnull().mean() * 100

faixas = {
    'Sem missing (0%)':        (missing_pct_numeric == 0).sum(),
    'Baixo missing (0–25%)':   ((missing_pct_numeric > 0) & (missing_pct_numeric <= 25)).sum(),
    'Médio missing (25–50%)':  ((missing_pct_numeric > 25) & (missing_pct_numeric <= 50)).sum(),
    'Alto missing (50–75%)':   ((missing_pct_numeric > 50) & (missing_pct_numeric <= 75)).sum(),
    'Muito alto missing (>75%)': (missing_pct_numeric > 75).sum(),
}

print('Distribuição de missing na tabela de transações:')
for faixa, count in faixas.items():
    bar = '█' * (count // 5)
    print(f'  {faixa:<30s}: {count:3d} colunas  {bar}')

In [ ]:
# Missings da tabela de identidade
missing_id = missing_summary(df_id)

print(f'Colunas com valores ausentes: {len(missing_id)} de {df_id.shape[1]}')
print('\nTop 15 colunas com mais missing na tabela de identidade:')
missing_id.head(15)

## 7. Estatísticas Descritivas das Variáveis Principais

Antes do merge, vamos inspecionar as variáveis mais interpretáveis.

In [ ]:
# TransactionDT — é em segundos desde uma referência, não um timestamp real
print('TransactionDT (segundos desde referência):')
print(f'  Mínimo: {df_trans["TransactionDT"].min():,}')
print(f'  Máximo: {df_trans["TransactionDT"].max():,}')
print(f'  Span em dias: {(df_trans["TransactionDT"].max() - df_trans["TransactionDT"].min()) / 86400:.1f}')

In [ ]:
# TransactionAmt — valor da transação em USD
print('TransactionAmt (USD):')
print(df_trans['TransactionAmt'].describe().to_string())
print(f'\nTransações acima de $1000: {(df_trans["TransactionAmt"] > 1000).sum():,} ({(df_trans["TransactionAmt"] > 1000).mean()*100:.1f}%)')

In [ ]:
# ProductCD — categoria do produto
print('ProductCD (categoria do produto):')
print(df_trans['ProductCD'].value_counts().to_string())
print('\nTaxa de fraude por ProductCD:')
print(df_trans.groupby('ProductCD')['isFraud'].mean().sort_values(ascending=False).map('{:.2%}'.format).to_string())

In [ ]:
# card4 e card6 — geralmente são tipo de cartão (Visa/Mastercard) e tipo (crédito/débito)
print('card4:')
print(df_trans['card4'].value_counts().head(10).to_string())
print('\ncard6:')
print(df_trans['card6'].value_counts().head(10).to_string())

In [ ]:
# P_emaildomain — domínio do e-mail do comprador
print('Top 15 domínios de e-mail do comprador (P_emaildomain):')
print(df_trans['P_emaildomain'].value_counts().head(15).to_string())

print('\nTaxa de fraude por domínio (top 10 por volume):')
top_domains = df_trans['P_emaildomain'].value_counts().head(10).index
fraud_by_domain = df_trans[df_trans['P_emaildomain'].isin(top_domains)].groupby('P_emaildomain')['isFraud'].mean()
print(fraud_by_domain.sort_values(ascending=False).map('{:.2%}'.format).to_string())

## 8. Merge das Tabelas

Agora fazemos o LEFT JOIN para criar o dataset completo. O campo de junção é `TransactionID`.

In [ ]:
df = merge_tables(df_trans, df_id)

# Verificação de integridade após o merge
print(f'\nTransações com dados de identidade: {df["DeviceType"].notna().sum():,} ({df["DeviceType"].notna().mean()*100:.1f}%)')
print(f'Transações sem dados de identidade: {df["DeviceType"].isna().sum():,} ({df["DeviceType"].isna().mean()*100:.1f}%)')

# Verificar se o target foi preservado
assert df['isFraud'].notna().all(), 'ERRO: isFraud tem NaN após o merge!'
assert len(df) == len(df_trans), 'ERRO: número de linhas mudou após o merge!'
print('\n✓ Integridade verificada: todas as transações preservadas, isFraud sem NaN')

In [ ]:
# Será que fraudes têm mais dados de identidade?
# (fraudes podem ocorrer com mais frequência de dispositivos conhecidos/rastreáveis)
print('% de transações COM dados de identidade, por classe:')
has_identity = df['DeviceType'].notna()
print(df.groupby('isFraud')[has_identity.name].apply(
    lambda x: f'{has_identity[x.index].mean()*100:.1f}%'
))

# Forma mais clara
for label, name in [(0, 'Legítimas'), (1, 'Fraudes')]:
    subset = has_identity[df['isFraud'] == label]
    print(f'  {name}: {subset.mean()*100:.1f}% têm dados de identidade')

## 9. Visão Geral do Dataset Final

Após o merge, temos nosso dataset de trabalho para as próximas fases.

In [ ]:
print('=== DATASET FINAL (após merge) ===')
print(f'Linhas  : {df.shape[0]:,}')
print(f'Colunas : {df.shape[1]}')
print(f'Memória : {df.memory_usage(deep=True).sum() / 1e9:.2f} GB')
print()

# Tipos de dados no dataset final
print('Tipos de dados:')
print(df.dtypes.value_counts().to_string())
print()

# Total de colunas numéricas vs categóricas
num_cols  = df.select_dtypes(include='number').columns.tolist()
cat_cols  = df.select_dtypes(include='object').columns.tolist()
print(f'Colunas numéricas  : {len(num_cols)}')
print(f'Colunas categóricas: {len(cat_cols)}')
print(f'Categóricas: {cat_cols}')

In [ ]:
# Salvar o dataset merged para uso nas próximas fases
# Salvamos em Parquet — formato binário muito mais eficiente que CSV para recarregar
# (leitura ~10x mais rápida, arquivo ~3x menor, preserva tipos de dados)
output_path = os.path.join(DATA_DIR, 'train_merged.parquet')
df.to_parquet(output_path, index=False)
print(f'Dataset salvo em: {output_path}')
print(f'Tamanho do arquivo: {os.path.getsize(output_path) / 1e6:.1f} MB')

## 10. Resumo da Fase 1

### O que aprendemos:

| Aspecto | Descoberta |
|---------|------------|
| **Volume** | 590k transações × 434 colunas após merge |
| **Target** | ~3.5% de fraudes — dataset fortemente desbalanceado |
| **Missing** | Grande volume de missing, especialmente em V1–V339 e variáveis de identidade |
| **Identidade** | Apenas ~24% das transações têm dados de identidade |
| **Variáveis** | Mix de numéricas (maioria) e categóricas (ProductCD, card4, card6, emails, device) |
| **Tempo** | ~6 meses de dados (TransactionDT em segundos) |

### O que faremos a seguir (Fase 2 — EDA):

- Visualizar a distribuição do desbalanceamento
- Analisar distribuições de TransactionAmt por classe (fraude vs legítimo)
- Explorar padrões temporais de fraude
- Identificar as variáveis mais correlacionadas com `isFraud`
- Examinar os missings com mais profundidade (missing é aleatório ou sistemático?)